# Valid Outcomes of Positive Support Size Five

In [1]:
import time
import numpy as np
import chipsplitting.hyperfield.utils as utils
from chipsplitting.hyperfield import (
    HyperfieldHomogeneousLinearSystem as HVLinearSystem,
)
from chipsplitting import pairing_matrix, print_matrix, PascalForm

## Proposition 6.3

In [2]:
# positive support size
support_size = 5
# hyperfield variety generated by all pascal equations
S = {}
for d in range(8, 42):
    start = time.time()
    base_types = ["diag", "row", "col"]
    A = [PascalForm(d, b, k).to_hyperfield() for b in base_types for k in range(d + 1)]
    linear_system = HVLinearSystem(A)
    solutions = linear_system.quick_solve_loop(support_size)
    S[d] = solutions
    print(f"d={d}, #solutions={len(solutions)}")
    end = time.time()
    print(f"Elapsed time: {end - start}")

d=8, #solutions=792
Elapsed time: 0.008839845657348633
d=9, #solutions=882
Elapsed time: 0.012243032455444336
d=10, #solutions=950
Elapsed time: 0.018864154815673828
d=11, #solutions=1084
Elapsed time: 0.03840899467468262
d=12, #solutions=1102
Elapsed time: 0.03932023048400879
d=13, #solutions=1212
Elapsed time: 0.048441171646118164
d=14, #solutions=1248
Elapsed time: 0.07161498069763184
d=15, #solutions=1400
Elapsed time: 0.0875248908996582
d=16, #solutions=1400
Elapsed time: 0.127579927444458
d=17, #solutions=1530
Elapsed time: 0.15761041641235352
d=18, #solutions=1553
Elapsed time: 0.2151811122894287
d=19, #solutions=1723
Elapsed time: 0.2461869716644287
d=20, #solutions=1710
Elapsed time: 0.3381941318511963
d=21, #solutions=1856
Elapsed time: 0.38674473762512207
d=22, #solutions=1863
Elapsed time: 0.5429949760437012
d=23, #solutions=2049
Elapsed time: 0.636674165725708
d=24, #solutions=2020
Elapsed time: 0.8447742462158203
d=25, #solutions=2182
Elapsed time: 1.0348758697509766
d=26

In [3]:
# lets store our result in a json file
# so we don't need to compute the variety again
import json

json_file_name = 'section7_hyperfield_variety.json'
with open(json_file_name, 'w') as json_file:
    json.dump(S, json_file, default=int)

print(f"JSON dump has been written to {json_file_name}")

JSON dump has been written to section7_hyperfield_variety.json


Now, we generated $S$. Every configuration in $S$ can correspond to some valid outcome.

We apply our second filter: the invertibility criterion.

In [4]:
# we load our variety from a json file
import json

json_file_name = 'section7_hyperfield_variety.json'
with open(json_file_name, 'r') as json_file:
    S = json.load(json_file)
    S = {int(key): value for key, value in S.items()}
print("Loaded variety S")

Loaded variety S


In [5]:
%%time
low_rank = []
for d in range(8,42):
    for config in S[d]:
        # support of negative and positive entries
        support = [(0,0)] + [utils.to_coordinate(c) for c in config]
        basis = list(range(d + 1))
        P = pairing_matrix(d, basis, support)
        rank = np.linalg.matrix_rank(P)

        if rank < len(support):
            low_rank.append(support)
            print(f"Pairing matrix (degree={d}, support={support}, basis={basis}) has rank: {rank}")

CPU times: user 2min 12s, sys: 406 ms, total: 2min 12s
Wall time: 2min 12s


**Conclusion**: For $8 \leq d \leq 41$ no valid outcome of degree $d$ exist.

## Proposition 6.11

In [6]:
%%time
d = 13
A = [
    PascalForm(d, "diag", 1), PascalForm(d, "diag", 2), PascalForm(d, "diag", 3), 
    PascalForm(d, "diag", d - 3), PascalForm(d, "diag", d - 2), PascalForm(d, "diag", d - 1),
    PascalForm(d, "row", 1), PascalForm(d, "row", 2), PascalForm(d, "row", 3),
    PascalForm(d, "col", 1), PascalForm(d, "col", 2), PascalForm(d, "col", 3),
    PascalForm(d, "row", d - 3), PascalForm(d, "row", d - 2), PascalForm(d, "row", d - 1), PascalForm(d, "row", d),
    PascalForm(d, "col", d - 3), PascalForm(d, "col", d - 2), PascalForm(d, "col", d - 1), PascalForm(d, "col", d),
]
A = [p.to_hyperfield().contract(4) for p in A]
linear_system = HVLinearSystem(A)
solutions_odd = linear_system.quick_solve_loop(5)
print(len(solutions_odd))

1265
CPU times: user 11.5 ms, sys: 1.24 ms, total: 12.7 ms
Wall time: 12.5 ms


In [7]:
%%time
d = 14
A = [
    PascalForm(d, "diag", 1), PascalForm(d, "diag", 2), PascalForm(d, "diag", 3), 
    PascalForm(d, "diag", d - 3), PascalForm(d, "diag", d - 2), PascalForm(d, "diag", d - 1),
    PascalForm(d, "row", 1), PascalForm(d, "row", 2), PascalForm(d, "row", 3),
    PascalForm(d, "col", 1), PascalForm(d, "col", 2), PascalForm(d, "col", 3),
    PascalForm(d, "row", d - 3), PascalForm(d, "row", d - 2), PascalForm(d, "row", d - 1), PascalForm(d, "row", d),
    PascalForm(d, "col", d - 3), PascalForm(d, "col", d - 2), PascalForm(d, "col", d - 1), PascalForm(d, "col", d),
]
A = [p.to_hyperfield().contract(4) for p in A]
linear_system = HVLinearSystem(A)
solutions_even = linear_system.quick_solve_loop(5)
print(len(solutions_even))

1283
CPU times: user 33.5 ms, sys: 1.33 ms, total: 34.9 ms
Wall time: 34.2 ms


In [8]:
unique_solutions = list(set([*solutions_odd, *solutions_even]))
len(unique_solutions)

2318

We see that $\Gamma^{even} \cup \Gamma^{odd}$ contains 2318 solutions.

In [9]:
def chi(config):
    t = [x for x in config]
    for x in [60, 61, 62, 63]:
        if x in t:
            t.remove(x)
            t.append(x - 4)
    t.sort()
    return tuple(set(t))    

In [10]:
# count the number of unique elements in Lambda
Lambda = set()
for config in unique_solutions:
    Lambda.add(chi(config))
len(Lambda)

2290

## Corollary 6.13

In [11]:
d0 = set([56, 57, 58, 59])
d1 = set([60, 61, 62, 63])
for config in unique_solutions:
	if set(config).intersection(d0) and set(config).intersection(d1):
		print(f"Config {config} has positive support 4")

Config (np.int64(3), np.int64(5), np.int64(12), np.int64(56), np.int64(60)) has positive support 4


In [12]:
Lambda.remove(chi((3,5,12,56, 60)))

In [13]:
len(Lambda)

2289

In [14]:
Lambda = list(Lambda)

## Reduction to 1107 Cases

### Code

In [15]:
def relcoord(index):
    """
    index: index in super contracted coordinate system
    """
    M = 500
    d = 1000
    indexes = { 
        'x': set(range(16)),
        'y': set(range(16, 32)),
        'z': set(range(32, 48)),
        'b': set(range(48, 52)),
        'c': set(range(52, 56)),
        'd': set(range(56, 60)),
    }
    if index in indexes['x']:
        col = index // 4
        row = index % 4
        return [(col, row)]
    elif index in indexes['y']:
        col = (index - 16) // 4
        row = (-3 + index % 4) - col + d
        return [(col, row)]
    elif index in indexes['z']:
        row = index % 4
        col = d + (-3 + (index - 32) // 4) - row
        return [(col, row)]
    elif index in indexes['b']:
        b_index = index - 48
        return [(M, b_index)] + [(d-6 + i , b_index) for i in range(3 - b_index)]
    elif index in indexes['c']:
        c_index = index - 52
        return [(c_index, M)] + [(c_index, d-6 + i) for i in range(3 - c_index)]
    elif index in indexes['d']:
        d_index = index - 56
        return [(M, M)] + [(M, d-6 + i) for i in range(3 - d_index)] + [(d-6 + i, M) for i in range(3 - d_index)]
    else:
        raise Value

def stringify_relcoord(x):
    if x == 500:
        return 'M'
    elif x == 1000:
        return 'd'
    elif x > 500:
        return f"d{x - 1000}"
    else:
        return x

def print_relcoord(coord):
    print([(stringify_relcoord(x), stringify_relcoord(y)) for (x,y) in coord])

def relsets(config):
    """
    Find all configurations in Z^Vd in relative coordinate system 
    that map to super_contracted_config under (contract'◦sign)
    """
    assert all([True if 0 <= x and x < 60 else False for x in config]), "Passed argument is not in valid super contracted form"

    rel_config = [relcoord(x) for x in config]
    accu = []
    res = []
    def dfs(index):
        if index >= len(rel_config):
            res.append(accu.copy())
            return
        for x in rel_config[index]:
            accu.append(x)
            dfs(index + 1)
            accu.pop()
    
    dfs(0)
    return res

In [16]:
# Test relsets
lambd = Lambda[100]

print(f"Super contracted configuration {Lambda[100]}")
print("Here is a list of configurations in Z^Vd that map to this super contracted configuration:")
print()
for r in relsets(lambd):
    print([(stringify_relcoord(x), stringify_relcoord(y)) for x,y in r])

print()
print("Configurations are given in relative coordinates.")

Super contracted configuration (np.int64(12), np.int64(45), np.int64(19), np.int64(20), 56)
Here is a list of configurations in Z^Vd that map to this super contracted configuration:

[(np.int64(3), np.int64(0)), ('d-1', np.int64(1)), (np.int64(0), 'd'), (np.int64(1), 'd-4'), ('M', 'M')]
[(np.int64(3), np.int64(0)), ('d-1', np.int64(1)), (np.int64(0), 'd'), (np.int64(1), 'd-4'), ('M', 'd-6')]
[(np.int64(3), np.int64(0)), ('d-1', np.int64(1)), (np.int64(0), 'd'), (np.int64(1), 'd-4'), ('M', 'd-5')]
[(np.int64(3), np.int64(0)), ('d-1', np.int64(1)), (np.int64(0), 'd'), (np.int64(1), 'd-4'), ('M', 'd-4')]
[(np.int64(3), np.int64(0)), ('d-1', np.int64(1)), (np.int64(0), 'd'), (np.int64(1), 'd-4'), ('d-6', 'M')]
[(np.int64(3), np.int64(0)), ('d-1', np.int64(1)), (np.int64(0), 'd'), (np.int64(1), 'd-4'), ('d-5', 'M')]
[(np.int64(3), np.int64(0)), ('d-1', np.int64(1)), (np.int64(0), 'd'), (np.int64(1), 'd-4'), ('d-4', 'M')]

Configurations are given in relative coordinates.


### Divide

In [17]:
SENTINEL_M = 500
SENTINEL_D = 1000
REL = [0,1,2,3,SENTINEL_M,SENTINEL_D-6,SENTINEL_D-5,SENTINEL_D-4,SENTINEL_D-3,SENTINEL_D-2,SENTINEL_D-1,SENTINEL_D]

def col(p):
    return p[0]

def row(p):
    return p[1]

def divide(relset):
    """
    Computes the lambda for the division step. 
    Returns lambda if a division was found, otherwise None is returned.

    Note that if None is returned, it does not necessarily mean that there is no division. 
    It just means that this (simple) algorithm could not find a division,
    but there may exist one, and we haven't found it.

    relset: support of a configuration in relative coordinates
    """
    M = 500
    d = 1000
    R = [0, 1, 2, 3, M, d-6, d-5, d-4, d-3, d-2, d-1, d] # relative coordinate system

    assert len(relset) == 6 #size of pos support + size of negative support = 6

    lambd = []
    col_start = 0
    for col_end in range(len(R)):
        num_cols = col_end - col_start + 1
        points_in_col = [p for p in relset if col(p) in range(R[col_start], R[col_end]+1)]
        num_points = len(points_in_col)

        assert not num_points or num_cols <= num_points

        if not num_points or num_cols == num_points or R[col_end] == M:
            lambd.append(num_points)
            col_start = col_end + 1
            
    if sum(lambd) != 6:
        return None
        
    return lambd

In [18]:
# Test divide
l = Lambda[400]
for relset in relsets(l):
    config = [(0,0)] + relset
    config_str = str(sorted([(stringify_relcoord(x), stringify_relcoord(y)) for x,y in config], key=lambda x: str(x[0])))
    lambd = divide(config)
    print(f"Computed lambda={lambd} for config {config_str}.")

Computed lambda=[3, 0, 2, 0, 0, 0, 0, 0, 0, 1] for config [(0, 0), (np.int64(0), np.int64(3)), (np.int64(1), 'd-1'), ('M', np.int64(1)), ('M', 'M'), ('d', np.int64(0))].
Computed lambda=[3, 0, 2, 0, 0, 0, 0, 0, 0, 1] for config [(0, 0), (np.int64(0), np.int64(3)), (np.int64(1), 'd-1'), ('M', np.int64(1)), ('M', 'd-6'), ('d', np.int64(0))].
Computed lambda=[3, 0, 2, 0, 0, 0, 0, 0, 0, 1] for config [(0, 0), (np.int64(0), np.int64(3)), (np.int64(1), 'd-1'), ('M', np.int64(1)), ('M', 'd-5'), ('d', np.int64(0))].
Computed lambda=[3, 0, 1, 1, 0, 0, 0, 0, 0, 1] for config [(0, 0), (np.int64(0), np.int64(3)), (np.int64(1), 'd-1'), ('M', np.int64(1)), ('d', np.int64(0)), ('d-6', 'M')].
Computed lambda=[3, 0, 1, 0, 1, 0, 0, 0, 0, 1] for config [(0, 0), (np.int64(0), np.int64(3)), (np.int64(1), 'd-1'), ('M', np.int64(1)), ('d', np.int64(0)), ('d-5', 'M')].
Computed lambda=[3, 0, 1, 1, 0, 0, 0, 0, 0, 1] for config [(0, 0), (np.int64(0), np.int64(3)), (np.int64(1), 'd-1'), ('M', 'M'), ('d', np.int6

### Conquer

In [19]:
SENTINEL_M = 500
SENTINEL_D = 1000

def col(p):
    return p[0]

def row(p):
    return p[1]

def succ(x):
    if x == 3:
        return SENTINEL_M
    if x == SENTINEL_M:
        return SENTINEL_D-6
    return x+1

def midpoint(a,b) -> list[int]:
    low, high = min(a,b), max(a,b)

    # low and high take concrete values
    # hence we can compute the midpoint exactly
    if low > SENTINEL_M or high < SENTINEL_M: 
        return [(low + high - 1) / 2]
        
    if high == SENTINEL_M:
        if low in [0,1]:
            return [2,3,SENTINEL_M]
        if low in [2,3]:
            return [3,SENTINEL_M]
    if low == SENTINEL_M:
        if high in [SENTINEL_D-6, SENTINEL_D-5]:
            return [SENTINEL_M]
        if high in [SENTINEL_D-4, SENTINEL_D-3]:
            return [SENTINEL_M,SENTINEL_D-6]
        if high in [SENTINEL_D-2, SENTINEL_D-1]:
            return [SENTINEL_M,SENTINEL_D-6,SENTINEL_D-5]       
        if high == SENTINEL_D:
            return [SENTINEL_M,SENTINEL_D-6,SENTINEL_D-5,SENTINEL_D-4]
    
    return [SENTINEL_M]
        
def conquer(relset):
    """
    Returns True if invertibility criterion can be applied, otherwise False
    """
    relset = sorted(relset, key=lambda x: col(x))
    length = len(relset)
    if length <= 2: # Proposition 4.18 and Proposition 4.19
        return True
    elif length == 3: 
        x, y, z = relset
        same_column = col(x) == col(y) and col(y) == col(z)
        
        if same_column: # Proposition 4.20
            return True

        if col(x) == col(y) and succ(col(x)) == col(z): # Proposition 4.21
            if row(z) not in midpoint(row(x),row(y)): # Proposition 6.24
                return True
            
        return False
    else:
        return False

### Apply

In [20]:
# helper function
def make_subrelset(relset, lambd):
    nonzero = [l for l in lambd if l > 0]
    begin = 0
    for n in nonzero:
        yield relset[begin:begin+n]
        begin += n

def apply_invertibility_criterion(relset):
    assert len(relset) == 6 #size of negative support + size of negative support = 6
    
    relset = sorted(relset, key=lambda x: col(x))
    lambd = divide(relset)
    conquer_success = all([conquer(subrelset) for subrelset in make_subrelset(relset, lambd)])

    return conquer_success

In [21]:
left_to_check = []
for l in Lambda:
    res_inv_crit = [apply_invertibility_criterion([(0,0)] + r) for r in relsets(l)]
    if not all(res_inv_crit):
        left_to_check.append(l)
len(left_to_check)

1107

## Reduction to 23 Cases

In [22]:
import chipsplitting.hyperfield.super_contraction_form as SCF

def apply_S3_action(sigma, config): 
    if sigma == "12":
        res = []
        for index in config:
            var = SCF.index_to_var(index)
            char = var[0]
            if char in ['x','y','z']:
                col = var[1]
                row = var[2]
                if char == 'x':
                    res.append(SCF.var_to_index(f"x{row}{col}"))
                elif char == 'y':
                    res.append(SCF.var_to_index(f"z{row}{col}"))
                else:
                    res.append(SCF.var_to_index(f"y{row}{col}"))
            elif char == 'b':
                row = var[1]
                res.append(SCF.var_to_index(f"c{row}"))
            elif char == 'c':
                col = var[1]
                res.append(SCF.var_to_index(f"b{col}"))
            elif char == 'd':
                res.append(index)
        assert len(res) == len(config)
        return tuple(sorted(res))
    elif sigma == "13":
        res = []
        for index in config:
            var = SCF.index_to_var(index)
            char = var[0]
            if char in ['x','y','z']:
                col = int(var[1])
                row = int(var[2])
                if char == 'x':
                    res.append(SCF.var_to_index(f"z{3 - col}{row}"))
                elif char == 'y':
                    res.append(SCF.var_to_index(f"y{3 - row}{3 - col}"))
                elif char == 'z':
                    res.append(SCF.var_to_index(f"x{3 - col}{row}"))
                else:
                    assert False
            elif char == 'b':
                res.append(index)
            elif char == 'c':
                col = var[1]
                res.append(SCF.var_to_index(f"d{col}"))
            elif char == 'd':
                col = var[1]
                res.append(SCF.var_to_index(f"c{col}"))
        assert len(res) == len(config)
        return tuple(sorted(res))
    elif sigma == 3:
        pass
    else:
        raise ValueError("Unknown sigma")

def config_with_zero(config):
    return [0] + list(config)

def any_relset_fails_invertibility_criterion(config):
    return not all([apply_invertibility_criterion(r) for r in relsets(config)])



In [23]:
# sigma defines the group action (12)
sigma = "12"
print("Apply symmetry and invertibility criterion on left_to_check...")
apply_symmetry = lambda config: apply_S3_action(sigma, config)
inv_relset_symmetry_predicate = lambda config: any_relset_fails_invertibility_criterion(apply_symmetry(config_with_zero(config)))
left_to_check = list(filter(inv_relset_symmetry_predicate, left_to_check))
print("Symmetry and invertbility criterion successfully applied.")
print(f"left_to_check contains {len(left_to_check)} configurations after applying a symmetry and the invertibility criterion.")

sigma = "13"
print("Apply symmetry and invertibility criterion on left_to_check...")
apply_symmetry = lambda config: apply_S3_action(sigma, config)
inv_relset_symmetry_predicate = lambda config: any_relset_fails_invertibility_criterion(apply_symmetry(config_with_zero(config)))
left_to_check = list(filter(inv_relset_symmetry_predicate, left_to_check))
print("Symmetry and invertbility criterion successfully applied.")
print(f"left_to_check contains {len(left_to_check)} configurations after applying a symmetry and the invertibility criterion.")

Apply symmetry and invertibility criterion on left_to_check...
Symmetry and invertbility criterion successfully applied.
left_to_check contains 547 configurations after applying a symmetry and the invertibility criterion.
Apply symmetry and invertibility criterion on left_to_check...
Symmetry and invertbility criterion successfully applied.
left_to_check contains 349 configurations after applying a symmetry and the invertibility criterion.


In [24]:
# after_symmetry holds the equivalence classes
after_symmetry = []

for config in left_to_check:
    to_add = True
    for other in after_symmetry:
        if apply_S3_action("12", config) == other or apply_S3_action("13", config) == other:
            to_add = False
            break
    if to_add:
        after_symmetry.append(config)

left_to_check = after_symmetry.copy()
print(f"After symmetry, {len(left_to_check)} configurations remain left to check.") 

After symmetry, 348 configurations remain left to check.


In [25]:
# remaining contains configurations that have at least nonzero b, c or d component
remaining = []

for config in left_to_check:
    vars = set([SCF.index_to_var(x) for x in config])
    # if configuration contains at least one b, c or d
    if vars.intersection(set(['b0', 'b1', 'b2', 'b3', 'c0', 'c1', 'c2', 'c3', 'd0', 'd1', 'd2', 'd3'])):
        remaining.append(config)

print(f"After the Hexagon criterion, {len(remaining)} configurations remain left to check.") 
print(f"Here are the remaining configurations.")
print()

for config in [[SCF.index_to_var(index) for index in sorted(config)] for config in remaining]:
    print(config)

After the Hexagon criterion, 23 configurations remain left to check.
Here are the remaining configurations.

['y03', 'z20', 'z22', 'z31', 'c1']
['y03', 'y12', 'y21', 'z30', 'b1']
['y03', 'y11', 'y13', 'z20', 'b1']
['y03', 'y13', 'y22', 'z20', 'b1']
['x01', 'x21', 'y13', 'z30', 'd1']
['y03', 'z10', 'z22', 'z31', 'c1']
['y02', 'y11', 'y13', 'z30', 'b1']
['y02', 'y13', 'y22', 'z30', 'b1']
['y02', 'z11', 'z30', 'z31', 'c1']
['y03', 'z22', 'z30', 'z31', 'c1']
['y01', 'y13', 'y22', 'z30', 'b1']
['y03', 'z11', 'z30', 'z31', 'c1']
['x03', 'x11', 'x30', 'z33', 'd0']
['x02', 'x21', 'y13', 'z30', 'd1']
['y03', 'z30', 'b1', 'c1', 'd1']
['y03', 'z11', 'z20', 'z31', 'c1']
['y03', 'y11', 'y13', 'z30', 'b1']
['x10', 'x12', 'y03', 'z31', 'd1']
['y02', 'z22', 'z30', 'z31', 'c1']
['y03', 'y13', 'y22', 'z30', 'b1']
['x12', 'x20', 'y03', 'z31', 'd1']
['x12', 'x21', 'y03', 'z30', 'd1']
['y03', 'z12', 'z21', 'z30', 'c1']
